In [ ]:
import puss
from puss import reaction_bools
from puss import single_reaction_bool
from puss import prepolymer_reaction
from puss import prepolymer_reaction_bools
file="data.csv"
prepolymer_reaction_bools(file)
reaction_bools(file)

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('data1.csv')
smiles_list = df.iloc[:, 5].astype(str).tolist()
print(f"✅ 成功读取 {len(smiles_list)} 个 SMILES")


descriptor_names = [desc[0] for desc in Descriptors._descList]
print(f"✅ 共 {len(descriptor_names)} 个 RDKit 描述符")

descriptor_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

def calc_descriptors(smiles):
 
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return [np.nan] * len(descriptor_names)
        return descriptor_calculator.CalcDescriptors(mol)
    except Exception as e:
        return [np.nan] * len(descriptor_names)


n_jobs =6  
print(f"⚙️ 开始并行计算，使用 {n_jobs if n_jobs>0 else '全部可用'} 核...")

results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
    delayed(calc_descriptors)(smi) for smi in tqdm(smiles_list, desc="计算中")
)


df_desc = pd.DataFrame(results, columns=descriptor_names)
print(f"✅ 特征矩阵形状: {df_desc.shape}")

df_out = pd.concat([df.reset_index(drop=True), df_desc.reset_index(drop=True)], axis=1)
df_out.to_csv("pu1.csv", index=False)

